In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

FINAL_OUTPUT_PATH = "pilot_llm_output.csv"
JSONL_LOG_PATH = "pilot_api_log.jsonl"
FIG_PATH = os.path.join("..", "figures", "fig1_distribution.png")

df_output = pd.read_csv(FINAL_OUTPUT_PATH)
print(f"Loaded {len(df_output)} pilot rows.")
df_output.head()

In [ ]:
total_rows = len(df_output)
invalid_rows = df_output[df_output["status"] == "INVALID"] if "status" in df_output.columns else pd.DataFrame()
invalid_count = len(invalid_rows)
invalid_ratio = invalid_count / total_rows if total_rows > 0 else 0
print(f"INVALID: {invalid_count}/{total_rows} ({invalid_ratio:.2%})")

In [ ]:
log_records = []
if os.path.exists(JSONL_LOG_PATH):
    with open(JSONL_LOG_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                log_records.append(json.loads(line))

df_logs = pd.DataFrame(log_records) if log_records else pd.DataFrame()
total_cost = df_logs["cost_usd"].sum() if "cost_usd" in df_logs.columns else 0.0
avg_latency = df_logs["latency_sec"].mean() if "latency_sec" in df_logs.columns else np.nan
print(f"Cost: ${total_cost:.6f} USD | Avg latency: {avg_latency:.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
if not df_logs.empty and "latency_sec" in df_logs.columns:
    sns.histplot(data=df_logs, x="latency_sec", kde=True, ax=axes[0], color="teal", bins=15)
if "status" in df_output.columns:
    sns.countplot(data=df_output, x="status", ax=axes[1], palette="Set2")
plt.tight_layout()
os.makedirs(os.path.dirname(FIG_PATH), exist_ok=True)
plt.savefig(FIG_PATH, bbox_inches="tight")
print("Saved", FIG_PATH)
plt.show()